# Projeto 2: Análise Exploratória & Tratamento de Dados de E-commerce

**Autor:** Ieda Oliveira  
**Objetivo:** Realizar a limpeza, tratamento de valores nulos/duplicados, engenharia de atributos e análise exploratória de dados (EDA) com visualizações estatísticas sobre a base transacional de e-commerce (Olist).

## 1. Importação das Bibliotecas e Carregamento dos Dados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuração de estilo visual
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 10

# Leitura dos datasets (ajuste os caminhos se necessário)
df_orders = pd.read_csv('olist_orders_dataset.csv')
df_items = pd.read_csv('olist_order_items_dataset.csv')
df_products = pd.read_csv('olist_products_dataset.csv')
df_customers = pd.read_csv('olist_customers_dataset.csv')

print("Arquivos carregados com sucesso!")

## 2. Inspeção Inicial e Tratamento de Nulos & Tipos de Dados

In [ ]:
# Filtrar apenas pedidos entregues/faturados
df_orders_delivered = df_orders[df_orders['order_status'] == 'delivered'].copy()

# Conversão de colunas temporais para Datetime
time_cols = ['order_purchase_timestamp', 'order_delivered_customer_date', 'order_estimated_delivery_date']
for col in time_cols:
    df_orders_delivered[col] = pd.to_datetime(df_orders_delivered[col])

# Tratamento de valores ausentes em categorias de produtos
df_products['product_category_name'] = df_products['product_category_name'].fillna('outros')

# Verificação de dados faltantes pós-limpeza
print("Nulos em Pedidos Entregues:")
print(df_orders_delivered[time_cols].isnull().sum())

## 3. Merge e Engenharia de Atributos (Feature Engineering)

In [ ]:
# Cruzamento das bases (Tabela Consolidada)
df_merged = df_orders_delivered.merge(df_items, on='order_id', how='inner') \
                               .merge(df_products[['product_id', 'product_category_name', 'product_weight_g']], on='product_id', how='left') \
                               .merge(df_customers[['customer_id', 'customer_city', 'customer_state']], on='customer_id', how='left')

# Cálculo do tempo real de entrega (em dias)
df_merged['delivery_time_days'] = (df_merged['order_delivered_customer_date'] - df_merged['order_purchase_timestamp']).dt.total_seconds() / (24 * 3600)

# Indicador de entrega com atraso (True / False)
df_merged['is_delayed'] = df_merged['order_delivered_customer_date'] > df_merged['order_estimated_delivery_date']

# Receita total do item
df_merged['total_item_value'] = df_merged['price'] + df_merged['freight_value']

print(f"Base consolidada gerada com {df_merged.shape[0]:,} linhas e {df_merged.shape[1]} colunas.")
df_merged.head(3)

## 4. Análise Estatística Descritiva

In [ ]:
# Resumo estatístico de variáveis numéricas essenciais
stats_cols = ['price', 'freight_value', 'total_item_value', 'delivery_time_days', 'product_weight_g']
df_merged[stats_cols].describe().round(2)

## 5. Visualizações e Descoberta de Padrões

In [ ]:
# 5.1 Distribuição de Preços (Removendo outliers extremos para visualização)
plt.figure(figsize=(10, 4))
sns.histplot(df_merged[df_merged['price'] < 500]['price'], bins=40, kde=True, color='#2b6cb0')
plt.title('Distribuição de Preço dos Produtos (Até R$ 500)', fontsize=12, fontweight='bold')
plt.xlabel('Preço (R$)')
plt.ylabel('Frequência')
plt.show()

In [ ]:
# 5.2 Matriz de Correlação entre Variáveis Numéricas
plt.figure(figsize=(8, 6))
corr_matrix = df_merged[['price', 'freight_value', 'delivery_time_days', 'product_weight_g']].corr()
sns.heatmap(corr_matrix, annot=True, cmap='Blues', fmt=".2f", cbar=True, square=True)
plt.title('Matriz de Correlação Linear (Pearson)', fontsize=12, fontweight='bold')
plt.show()

In [ ]:
# 5.3 Top 10 Categorias por Faturamento
top_categories = df_merged.groupby('product_category_name')['price'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(10, 5))
sns.barplot(x=top_categories.values, y=top_categories.index, palette='Blues_r')
plt.title('Top 10 Categorias por Volume Total de Vendas (R$)', fontsize=12, fontweight='bold')
plt.xlabel('Faturamento Total (R$)')
plt.ylabel('Categoria')
plt.show()

In [ ]:
# 5.4 Prazo Médio de Entrega por Estado (Top 10 Maiores Estados em Volume)
top_states = df_merged['customer_state'].value_counts().head(10).index
df_top_states = df_merged[df_merged['customer_state'].isin(top_states)]
state_delivery = df_top_states.groupby('customer_state')['delivery_time_days'].median().sort_values()

plt.figure(figsize=(10, 4))
sns.barplot(x=state_delivery.index, y=state_delivery.values, color='#3182ce')
plt.title('Mediana do Prazo de Entrega (Dias) por Estado (Top 10 em Vendas)', fontsize=12, fontweight='bold')
plt.xlabel('UF')
plt.ylabel('Dias de Entrega (Mediana)')
plt.show()

## 6. Principais Conclusões e Insights
- **Concentração de Ticket:** A maioria esmagadora das vendas se concentra em tíquetes abaixo de R$ 150,00, com forte cauda longa de produtos premium.
- **Logística e Peso:** O valor do frete (`freight_value`) possui forte correlação positiva com o peso do produto (`product_weight_g`).
- **Disparidade Regional de Prazos:** Estados das regiões Sul e Sudeste (SP, PR, RJ, MG) apresentam medianas de entrega entre 8 e 12 dias, enquanto estados mais afastados dos centros de distribuição sofrem com prazos sensivelmente maiores.